# NHANES Hv2 Colab Experiment

This notebook only runs experiment code.

It does not fabricate results.

It does not generate thesis Chapter 4 text.

本 notebook 只运行实验代码，不伪造实验结果，也不生成论文第4章正文。


In [ ]:
# 中文：挂载 Google Drive
# English: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 中文：克隆或更新 GitHub 仓库，并切换到仓库根目录
# English: Clone or update the GitHub repository, then switch to the repository root
import os
import subprocess

REPO_URL = "https://github.com/TiANLANG777/nhanes_2021_2023_health_index_experiment.git"
REPO_DIR = "/content/nhanes_2021_2023_health_index_experiment"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Current working directory: {os.getcwd()}")


In [ ]:
# 中文：从仓库根目录安装依赖
# English: Install dependencies from the repository root
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements_colab.txt"], cwd=REPO_DIR, check=True)


In [ ]:
# 中文：定义固定路径并创建输出子文件夹
# English: Define fixed paths and create output subfolders
import os

PROJECT_DIR = "/content/drive/MyDrive/nhanes_2021_2023_health_index_experiment"
DATA_DIR = PROJECT_DIR + "/data"
OUTPUT_DIR = PROJECT_DIR + "/outputs"

FULL_FEATURE_PATH = DATA_DIR + "/adult_full_feature_set_v2.csv"
REDUCED_FEATURE_PATH = DATA_DIR + "/adult_reduced_feature_set_v2.csv"
TARGET_PATH = DATA_DIR + "/adult_targets_v2.csv"

for required_path in [FULL_FEATURE_PATH, REDUCED_FEATURE_PATH, TARGET_PATH]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f"Missing required file: {required_path}")

for subdir in ["tables", "figures", "models", "reports"]:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

print(PROJECT_DIR)
print(DATA_DIR)
print(OUTPUT_DIR)


In [ ]:
# 中文：从仓库根目录运行数据完整性与泄漏检查
# English: Run data integrity and leakage checks from the repository root
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable,
    "scripts/01_data_check.py",
    "--data-dir",
    DATA_DIR,
    "--output-dir",
    OUTPUT_DIR,
], cwd=REPO_DIR, check=True)

with open(os.path.join(OUTPUT_DIR, "data_check", "data_check_report.md"), "r", encoding="utf-8") as handle:
    print(handle.read())


In [ ]:
# 中文：从仓库根目录运行 H_v2 回归模型
# English: Run H_v2 regression models from the repository root
import os
import subprocess
import sys

os.chdir(REPO_DIR)
for feature_set in ["full", "reduced"]:
    subprocess.run([
        sys.executable,
        "scripts/02_train_hv2_regression.py",
        "--feature-set",
        feature_set,
        "--data-dir",
        DATA_DIR,
        "--output-dir",
        OUTPUT_DIR,
    ], cwd=REPO_DIR, check=True)


In [ ]:
# 中文：从仓库根目录运行年龄组稳定性分析
# English: Run age-group stability analysis from the repository root
import os
import subprocess
import sys

os.chdir(REPO_DIR)
for feature_set in ["full", "reduced"]:
    subprocess.run([
        sys.executable,
        "scripts/03_age_group_stability.py",
        "--feature-set",
        feature_set,
        "--model",
        "random_forest",
        "--data-dir",
        DATA_DIR,
        "--output-dir",
        OUTPUT_DIR,
    ], cwd=REPO_DIR, check=True)


In [ ]:
# 中文：从仓库根目录运行 H_v1 敏感性分析
# English: Run H_v1 sensitivity analysis from the repository root
import os
import subprocess
import sys

os.chdir(REPO_DIR)
for feature_set in ["full", "reduced"]:
    subprocess.run([
        sys.executable,
        "scripts/04_hv1_sensitivity.py",
        "--feature-set",
        feature_set,
        "--model",
        "random_forest",
        "--data-dir",
        DATA_DIR,
        "--output-dir",
        OUTPUT_DIR,
    ], cwd=REPO_DIR, check=True)


In [ ]:
# 中文：从仓库根目录运行 SHAP 分析，并将其视为相关性解释而非因果解释
# English: Run SHAP analysis from the repository root and treat it as an association-based explanation, not a causal explanation
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable,
    "scripts/05_shap_analysis.py",
    "--feature-set",
    "full",
    "--data-dir",
    DATA_DIR,
    "--output-dir",
    OUTPUT_DIR,
], cwd=REPO_DIR, check=True)


In [ ]:
# 中文：从仓库根目录生成实验总结
# English: Generate the experiment summary from the repository root
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable,
    "scripts/06_generate_experiment_summary.py",
    "--output-dir",
    OUTPUT_DIR,
], cwd=REPO_DIR, check=True)

with open(os.path.join(OUTPUT_DIR, "experiment_summary", "experiment_summary.md"), "r", encoding="utf-8") as handle:
    print(handle.read())
